In [0]:
# Questo notebook aggiorna giornalmente la tabella Silver energia.
#
# Legge dalla Bronze ENTSO-E:
# progetto_meteo.bronze.dati_entsoe
#
# Prende solo il giorno appena concluso, cioè lo stesso giorno scaricato da:
# Streaming_entsoe_giornaliero
#
# Poi aggrega i dati per:
# - Area
# - Data
# - Ora
#
# e aggiorna la tabella Silver:
# progetto_meteo.silver.energy_block
#
# La tabella energy_block contiene le tre grandezze energetiche finali del progetto:
# - Solare
# - Idrico
# - Eolico
#
# Questo notebook viene usato nel JOB_AGGIORNAMENTO_GIORNALIERO,
# subito dopo Streaming_entsoe_giornaliero.

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from pyspark.sql import functions as F


# Imposto catalogo e tabelle del progetto.
catalogo = "progetto_meteo"

tabella_entsoe = f"{catalogo}.bronze.dati_entsoe"
tabella_energy_block = f"{catalogo}.silver.energy_block"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Calcolo il giorno appena concluso.
# Questo deve essere lo stesso giorno scaricato da Streaming_entsoe_giornaliero.
oggi = datetime.now(ZoneInfo("Europe/Rome")).date()
giorno_da_consolidare = oggi - timedelta(days=1)


# Converto la data nel formato usato da energy_block.
# In energy_block la colonna Data è una stringa nel formato dd/MM/yyyy.
data_energy_block = giorno_da_consolidare.strftime("%d/%m/%Y")

print(f"Giorno ENTSO-E da consolidare: {giorno_da_consolidare}")
print(f"Data energy_block: {data_energy_block}")


# Leggo solo i dati ENTSO-E del giorno appena concluso.
# La Bronze contiene la Data come tipo DATE.
df_entsoe_giorno = (
    spark.table(tabella_entsoe)
    .filter(F.col("Data") == F.lit(giorno_da_consolidare))
)


# Controllo che ci siano dati da aggregare.
# Se la Bronze non contiene il giorno richiesto, interrompo il notebook.
righe_bronze = df_entsoe_giorno.count()

if righe_bronze == 0:
    raise Exception(f"Nessun dato trovato in {tabella_entsoe} per il giorno {giorno_da_consolidare}. Esegui prima Streaming_entsoe_giornaliero.")


# Aggrego il giorno per Area, Data e Ora.
# Porto i tipi di produzione ENTSO-E nelle tre colonne finali del progetto:
# - Solare
# - Idrico
# - Eolico
df_energy_block_giorno = (
    df_entsoe_giorno
    .groupBy(
        "Area",
        "Data",
        "Ora"
    )
    .agg(
        F.round(
            F.sum(
                F.when(
                    F.col("Tipo_Produzione") == "Solar",
                    F.col("Produzione")
                )
            ),
            2
        ).alias("Solare"),

        F.round(
            F.sum(
                F.when(
                    F.col("Tipo_Produzione").isin(
                        "Hydro Run-of-river and pondage",
                        "Hydro Water Reservoir"
                    ),
                    F.col("Produzione")
                )
            ),
            2
        ).alias("Idrico"),

        F.round(
            F.sum(
                F.when(
                    F.col("Tipo_Produzione").isin(
                        "Wind Offshore",
                        "Wind Onshore"
                    ),
                    F.col("Produzione")
                )
            ),
            2
        ).alias("Eolico")
    )
    .fillna({
        "Solare": 0.0,
        "Idrico": 0.0,
        "Eolico": 0.0
    })
    .withColumn("Data", F.date_format("Data", "dd/MM/yyyy"))
    .select(
        "Area",
        "Data",
        "Ora",
        "Solare",
        "Idrico",
        "Eolico"
    )
)


# Controllo che l'aggregazione abbia prodotto righe.
# Se non ci sono righe aggregate, non aggiorno energy_block.
righe_energy_block = df_energy_block_giorno.count()

if righe_energy_block == 0:
    raise Exception("Aggregazione ENTSO-E giornaliera completata, ma nessuna riga prodotta. Non aggiorno energy_block.")


# Sovrascrivo solo il giorno appena concluso in energy_block.
# Non ricostruisco tutta la Silver energia.
predicato_silver = f"Data = '{data_energy_block}'"

df_energy_block_giorno.write.mode("overwrite").format("delta").option("replaceWhere", predicato_silver).saveAsTable(
    tabella_energy_block
)


# Controllo finale.
# Mostro le righe appena consolidate nella Silver energia.
display(
    df_energy_block_giorno
    .orderBy("Area", "Data", "Ora")
)


# Stampo un riepilogo finale del patch energia.
print("Patcher Energy Block completato.")
print(f"Giorno aggiornato in energy_block: {data_energy_block}")
print(f"Righe Bronze ENTSO-E lette: {righe_bronze}")
print(f"Righe scritte in energy_block: {righe_energy_block}")